# PromptAlign G0: GitHub → Kaggle → VS Code
Bật Internet và GPU. Restart kernel rồi Run All. Gửi lại g0-*.zip và notebook có output. Đây là kiểm tra môi trường, chưa huấn luyện GenAI-Bench. Đổi DEVICE thành cpu nếu không có GPU.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/hoan25/BaiqiL-GenAI-Bench.git"
# Dùng đúng commit của lần chạy G0 trước để tái lập kết quả.
# Nếu muốn lấy mã mới nhất, đổi thành "main" và ghi nhận commit mới ở output.
GIT_REF = "f48f135752e769fa5025a39896fcbb0475900e05"
DEVICE = "cuda"              # đổi thành "cpu" nếu không được cấp GPU
PRIVATE_REPO = False          # True chỉ khi repo private
WORKDIR = Path("/kaggle/working")
ROOT = WORKDIR / "promptalign-ai"
ENV_DIR = WORKDIR / ".promptalign-g0-venv"
ENV_PYTHON = ENV_DIR / "bin/python"

assert DEVICE in {"cuda", "cpu"}
print("Python kernel:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Thiết bị yêu cầu:", DEVICE)
subprocess.run([sys.executable, "-m", "pip", "--version"], check=True)

if DEVICE == "cuda":
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "Chưa phát hiện GPU; kiểm tra Accelerator trong Settings")


Python kernel: /usr/bin/python3
Python version: 3.13.15
Thiết bị yêu cầu: cuda
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
GPU: GPU 0: Tesla T4 (UUID: GPU-52f695ef-b269-dde8-2237-25b657e5d2fa)
GPU 1: Tesla T4 (UUID: GPU-e9c2d397-d60b-7a0d-d2e3-84ad225f215b)


In [2]:
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    if PRIVATE_REPO:
        import base64
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authorization = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": "AUTHORIZATION: basic " + authorization,
        })

    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], env=env, check=True)
    if not (ROOT / ".git").exists():
        raise RuntimeError(f"Đã tồn tại {ROOT}, nhưng không phải Git repository")

    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=ROOT, text=True
    ).strip()
    if origin != REPO_URL:
        raise RuntimeError(f"Repository không khớp (remote: {origin})")

    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=ROOT, text=True
    ).strip()
    if dirty:
        raise RuntimeError("Git working tree có thay đổi; hãy giữ lại thay đổi trước khi checkout")

    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=ROOT, env=env, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
finally:
    for key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
        env.pop(key, None)
    if PRIVATE_REPO:
        del token, authorization

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Repository:", REPO_URL)
print("Commit:", COMMIT)


Cloning into '/kaggle/working/promptalign-ai'...


Repository: https://github.com/hoan25/BaiqiL-GenAI-Bench.git
Commit: f48f135752e769fa5025a39896fcbb0475900e05


From https://github.com/hoan25/BaiqiL-GenAI-Bench
 * branch            f48f135752e769fa5025a39896fcbb0475900e05 -> FETCH_HEAD
HEAD is now at f48f135 Record GitHub handoff and pending Kaggle G0 validation


In [3]:
if ENV_DIR.exists():
    shutil.rmtree(ENV_DIR)

# Tránh ensurepip, vì trong môi trường Kaggle đã xác nhận module này không có.
subprocess.run(
    [sys.executable, "-m", "venv", "--without-pip", str(ENV_DIR)],
    check=True,
)
print("Tạo venv không dùng ensurepip: PASS")

# pip gốc đã hoạt động; cài riêng pip vào môi trường mới bằng --python.
subprocess.run(
    [sys.executable, "-m", "pip", "--python", str(ENV_DIR), "install", "pip"],
    check=True,
)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "--version"], check=True)
print("RESULT: VENV READY")


Tạo venv không dùng ensurepip: PASS


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.8 MB/s eta 0:00:00
pip 26.2.1 from /kaggle/working/.promptalign-g0-venv/lib/python3.13/site-packages/pip (python 3.13)
RESULT: VENV READY


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


In [4]:
if not (ROOT / "scripts" / "kaggle_g0.py").exists():
    raise FileNotFoundError("Không thấy scripts/kaggle_g0.py trong repository")
if not ENV_PYTHON.exists():
    raise RuntimeError("Chưa có venv G0. Hãy chạy Cell 3 trước")

archives_before = {p.name for p in WORKDIR.glob("g0-*.zip")}
G0_RUN = subprocess.run(
    [sys.executable, "-u", str(ROOT / "scripts" / "kaggle_g0.py"), "--device", DEVICE],
    cwd=ROOT,
    check=False,
)
G0_RETURN_CODE = G0_RUN.returncode
NEW_ARCHIVES = sorted(
    (p for p in WORKDIR.glob("g0-*.zip") if p.name not in archives_before),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
G0_ARCHIVE = NEW_ARCHIVES[0] if NEW_ARCHIVES else None
print("G0 return code:", G0_RETURN_CODE)
print("ZIP phiên chạy này:", str(G0_ARCHIVE) if G0_ARCHIVE else "CHƯA TẠO ĐƯỢC")


Running install_torch; log: /kaggle/working/g0-20261019T175231Z/install_torch.log
Running install_dependencies; log: /kaggle/working/g0-20261019T175231Z/install_dependencies.log
Running download_clip; log: /kaggle/working/g0-20261019T175231Z/download_clip.log
Running checks; log: /kaggle/working/g0-20261019T175231Z/checks.log
RESULT: PASS
SEND BACK: /kaggle/working/g0-20261019T175231Z.zip
G0 return code: 0
ZIP phiên chạy này: /kaggle/working/g0-20261019T175231Z.zip


In [5]:
import json
import zipfile
from IPython.display import FileLink, display

if G0_ARCHIVE is None:
    print("KHÔNG CÓ ZIP mới. Hãy gửi traceback / output của Cell 4 để chẩn đoán.")
else:
    required_when_pass = [
        "provenance.json", "result.json", "checks/environment.json",
        "checks/pytest.log", "checks/commands.json", "checks/clip_cpu.json",
        "checks/clip_auto.json", "checks/versions.log",
    ]
    with zipfile.ZipFile(G0_ARCHIVE) as z:
        names = set(z.namelist())
        print("ZIP:", G0_ARCHIVE.name)
        print("Dung lượng (bytes):", G0_ARCHIVE.stat().st_size)
        print("Danh sách file:")
        for name in sorted(names):
            print(" -", name)

        if "result.json" in names:
            result = json.loads(z.read("result.json"))
            print("\nRESULT:", result.get("status", "KHÔNG RÕ"))
            if result.get("status") == "PASS":
                missing = [name for name in required_when_pass if name not in names]
                print("Thiếu file bắt buộc:", missing if missing else "KHÔNG")

        if "checks/clip_auto.json" in names:
            clip_auto = json.loads(z.read("checks/clip_auto.json"))
            print("\nCLIP auto device:", clip_auto.get("device"))
            if DEVICE == "cuda" and clip_auto.get("device") != "cuda":
                print("CẢNH BÁO: yêu cầu CUDA nhưng kết quả chưa ghi device=cuda")

        if "error.txt" in names:
            print("\n=== error.txt ===")
            print(z.read("error.txt").decode("utf-8", errors="replace"))
            # In tối đa 40 dòng cuối của log ở bước lỗi gần nhất.
            stages = ["create_venv", "install_torch", "install_dependencies", "download_clip", "checks"]
            logs = [f"{stage}.log" for stage in stages if f"{stage}.log" in names]
            if logs:
                last_log = logs[-1]
                print(f"\n=== 40 dòng cuối {last_log} ===")
                print("\n".join(z.read(last_log).decode("utf-8", errors="replace").splitlines()[-40:]))

    print("\nTải ZIP từ /kaggle/working trong tab Output sau khi Save Version.")
    # Liên kết này có thể 404 trên một số phiên Kaggle; khi đó dùng tab Output.
    os.chdir(WORKDIR)
    display(FileLink(G0_ARCHIVE.name))


ZIP: g0-20261019T175231Z.zip
Dung lượng (bytes): 9416
Danh sách file:
 - checks.log
 - checks/
 - checks/clip_auto.json
 - checks/clip_auto.log
 - checks/clip_cpu.json
 - checks/clip_cpu.log
 - checks/commands.json
 - checks/environment.json
 - checks/environment.log
 - checks/pytest.log
 - checks/versions.log
 - clip_download.json
 - download_clip.log
 - install_dependencies.log
 - install_torch.log
 - provenance.json
 - result.json

RESULT: PASS
Thiếu file bắt buộc: KHÔNG

CLIP auto device: cuda

Tải ZIP từ /kaggle/working trong tab Output sau khi Save Version.


/kaggle/working/g0-20261019T175231Z.zip